# Round45 User Variants

Testa os prompts indicados pelo utilizador e variantes controladas, mantendo a melhor seed por metrica de cada classe.

Modo esperado: prompt search pequeno, sem variacao de seed. A seed usada e a seed derivada do nome do ficheiro: 1159, 7836 ou 9338.

In [1]:
from pathlib import Path
import json
import os
import subprocess
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src" / "search_seed_sweep.py").exists():
    PROJECT_ROOT = Path("C:\\Users\\tugap\\Desktop\\Universidade\\Masters2\u00baAno\\IAG\\ProjetoCunha\\Projeto2")

SRC_DIR = PROJECT_ROOT / "src"
PROMPT_BANK = PROJECT_ROOT / "prompts" / "refinement_round45_user_variants.json"
SEED_MAP = PROJECT_ROOT / "prompts" / "round43_best_metric_seeds.json"
TARGETS_DIR = PROJECT_ROOT / "TP2-students" / "students" / "tp2-chosen"
OUTPUT_DIR = PROJECT_ROOT / "TP2-students" / "students" / "outputs"

PYTHON_CANDIDATES = [
    PROJECT_ROOT / ".venv_win" / "Scripts" / "python.exe",
    Path("C:\\Users\\tugap\\Desktop\\Universidade\\Masters2\u00baAno\\IAG\\Projeto 2\\.venv\\Scripts\\python.exe"),
    Path(sys.executable),
]

def has_module(python_exe, module_name):
    if not Path(python_exe).exists():
        return False
    result = subprocess.run(
        [str(python_exe), "-c", f"import {module_name}"],
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True,
        encoding="utf-8",
        errors="replace",
    )
    return result.returncode == 0

PYTHON_EXE = None
for candidate in PYTHON_CANDIDATES:
    if has_module(candidate, "diffusers"):
        PYTHON_EXE = candidate
        break

if PYTHON_EXE is None:
    raise RuntimeError("No Python with diffusers found. Create .venv_win and install requirements.txt")

print("Project root:", PROJECT_ROOT)
print("Render/search Python:", PYTHON_EXE)
assert (SRC_DIR / "generate_round45_user_variants.py").exists()
assert (SRC_DIR / "search_seed_sweep.py").exists()
assert SEED_MAP.exists()


Project root: c:\Users\tugap\Desktop\Universidade\Masters2ºAno\IAG\ProjetoCunha\Projeto2
Render/search Python: C:\Users\tugap\Desktop\Universidade\Masters2ºAno\IAG\Projeto 2\.venv\Scripts\python.exe


## Configuracao

In [2]:
config = {
    "identity": "round45_user_variants",
    "prompt_source": "all",
    "seed_offsets": [0],
    "top_k": 10,
}

print(json.dumps(config, indent=2))

{
  "identity": "round45_user_variants",
  "prompt_source": "all",
  "seed_offsets": [
    0
  ],
  "top_k": 10
}


## Gerar banco de prompts

In [3]:
cmd = [str(PYTHON_EXE), str(SRC_DIR / "generate_round45_user_variants.py"), "--output", str(PROMPT_BANK)]
print("Running:", " ".join(cmd))
subprocess.run(cmd, cwd=PROJECT_ROOT, check=True)
data = json.loads(PROMPT_BANK.read_text(encoding="utf-8"))
print({target: len(entries) for target, entries in data.items()})
print("total renders:", sum(len(entries) for entries in data.values()) * len(config["seed_offsets"]))

Running: C:\Users\tugap\Desktop\Universidade\Masters2ºAno\IAG\Projeto 2\.venv\Scripts\python.exe c:\Users\tugap\Desktop\Universidade\Masters2ºAno\IAG\ProjetoCunha\Projeto2\src\generate_round45_user_variants.py --output c:\Users\tugap\Desktop\Universidade\Masters2ºAno\IAG\ProjetoCunha\Projeto2\prompts\refinement_round45_user_variants.json
{'1159_25.png': 5, '1159_29.png': 5, '1159_3.png': 6, '1159_7.png': 5, '7836.png': 5, '9338.png': 5}
total renders: 31


## Correr fixed-seed micro search

In [4]:
args = [
    str(PYTHON_EXE),
    str(SRC_DIR / "search_seed_sweep.py"),
    "--prompts", str(PROMPT_BANK),
    "--seed-map", str(SEED_MAP),
    "--targets", str(TARGETS_DIR),
    "--output-dir", str(OUTPUT_DIR),
    "--identity", config["identity"],
    "--prompt-source", config["prompt_source"],
    "--top-k", str(config["top_k"]),
    "--seed-offsets", *[str(seed) for seed in config["seed_offsets"]],
    "--offline",
    "--disable-progress-bar",
    "--maxstack-scoring",
]

env = os.environ.copy()
env["PYTHONIOENCODING"] = "utf-8"

print("Running:")
print(" ".join(args))
process = subprocess.Popen(
    args,
    cwd=PROJECT_ROOT,
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    encoding="utf-8",
    errors="replace",
    bufsize=1,
)
for line in process.stdout:
    print(line, end="")
return_code = process.wait()
if return_code != 0:
    raise RuntimeError(f"Round43 failed with exit code {return_code}")
print("Finished successfully")

Running:
C:\Users\tugap\Desktop\Universidade\Masters2ºAno\IAG\Projeto 2\.venv\Scripts\python.exe c:\Users\tugap\Desktop\Universidade\Masters2ºAno\IAG\ProjetoCunha\Projeto2\src\search_seed_sweep.py --prompts c:\Users\tugap\Desktop\Universidade\Masters2ºAno\IAG\ProjetoCunha\Projeto2\prompts\refinement_round45_user_variants.json --seed-map c:\Users\tugap\Desktop\Universidade\Masters2ºAno\IAG\ProjetoCunha\Projeto2\prompts\round43_best_metric_seeds.json --targets c:\Users\tugap\Desktop\Universidade\Masters2ºAno\IAG\ProjetoCunha\Projeto2\TP2-students\students\tp2-chosen --output-dir c:\Users\tugap\Desktop\Universidade\Masters2ºAno\IAG\ProjetoCunha\Projeto2\TP2-students\students\outputs --identity round45_user_variants --prompt-source all --top-k 10 --seed-offsets 0 --offline --disable-progress-bar --maxstack-scoring
Couldn't connect to the Hub: Cannot reach https://huggingface.co/api/models/SimianLuo/LCM_Dreamshaper_v7: offline mode is enabled. To disable it, please unset the `HF_HUB_OFFLINE

## Ver resultados

In [5]:
run_dirs = sorted(
    [path for path in OUTPUT_DIR.glob(f"*_{config['identity']}") if path.is_dir()],
    key=lambda path: path.stat().st_mtime,
    reverse=True,
)
if not run_dirs:
    print("No run directory found")
else:
    latest = run_dirs[0]
    print("Latest run:", latest)
    print("Contact sheet:", latest / f"contact_sheet_top{config['top_k']}_seed_sweep.jpg")
    print("CSV:", latest / f"top{config['top_k']}_seed_sweep.csv")
    for item in sorted(latest.glob("*")):
        print(item.name)

Latest run: c:\Users\tugap\Desktop\Universidade\Masters2ºAno\IAG\ProjetoCunha\Projeto2\TP2-students\students\outputs\20260603-215547_round45_user_variants
Contact sheet: c:\Users\tugap\Desktop\Universidade\Masters2ºAno\IAG\ProjetoCunha\Projeto2\TP2-students\students\outputs\20260603-215547_round45_user_variants\contact_sheet_top10_seed_sweep.jpg
CSV: c:\Users\tugap\Desktop\Universidade\Masters2ºAno\IAG\ProjetoCunha\Projeto2\TP2-students\students\outputs\20260603-215547_round45_user_variants\top10_seed_sweep.csv
1159_25
1159_25_seed_sweep_metrics.csv
1159_29
1159_29_seed_sweep_metrics.csv
1159_3
1159_3_seed_sweep_metrics.csv
1159_7
1159_7_seed_sweep_metrics.csv
7836
7836_seed_sweep_metrics.csv
9338
9338_seed_sweep_metrics.csv
contact_sheet_top10_seed_sweep.jpg
seed_sweep_metrics.csv
summary.json
top10_seed_sweep.csv
